# CBF-QP vs MPC-CBF — 100 randomised scenarios

The claim is **not** "MPC-CBF is better"; it is **"here is exactly where each one fails, and
why"**. Both controllers share the identical plant (exact-ZOH double integrator), the identical
cost (repo default weights $q=[10,10,1,1]$, $r=[1,1]$, $q_f=10q$) and the identical barrier
(codegen `barrier_expression`). The only difference is the horizon:

| controller | horizon | what it is |
|---|---|---|
| CBF-QP (myopic filter) | $N=1$ | the horizon-1 specialisation of the MPC-CBF: a one-step lookahead that picks the tracking-optimal input subject to the one-step DCBF — the discrete-time analogue of the one-step CBF-QP filter on a nominal tracking law (see `docs/derivations/mpc_cbf_unified_formulation.tex`, specialisation table: $N=1 \to$ CBF-QP) |
| MPC-CBF | $N=15$ | the full lookahead; DCBF rows on every prediction stage |

Failure modes this notebook measures (each becomes an `assert`):

* **Solve time.** The $N=1$ filter solves a 2-variable QP; $N=15$ solves 30. Measured below:
  ~20× per successful control decision, inside the 10–50× band the design doc predicts.
* **CBF-QP's failure mode.** One step of lookahead cannot see the obstacle closing in. When the
  one-step DCBF becomes infeasible the loop brakes (`u=0` fallback, the node's behaviour) and the
  ego coasts — often into the obstacle: 49/100 runs violate $h<0$ in the measured grid.
* **MPC-CBF's failure mode.** The horizon-15 constraint set (DCBF rows at every stage) can become
  infeasible from a state the horizon-1 problem still solves — the feasible set shrinks with the
  horizon (the feasibility-recovery notebook measures this). The planner commits early, and the
  commitment can strand it short of the goal: 8/100 runs never reach the goal while the myopic
  filter does.

All scenarios are static-obstacle (the repo's 2-D fixture scenario; the dynamic-obstacle case is
exercised by the `quadrotor_dynamic_obstacle` launch). Scenarios are persisted to
`analysis/scenarios.json` with the fixed seed, so the numbers are re-checkable and the GIF at the
bottom reuses the same scenarios.


In [ ]:
# --- Imports + determinism ------------------------------------------------
import sys
import time
import json
import os
from pathlib import Path

import matplotlib
matplotlib.use("Agg")  # headless; CI has no display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def find_repo_root(start: Path | None = None) -> Path:
    d = (start or Path.cwd()).resolve()
    for _ in range(6):
        if (d / "codegen").is_dir():
            return d
        if d.parent == d:
            break
        d = d.parent
    raise RuntimeError("repo root not found")


REPO_ROOT = find_repo_root()
sys.path.insert(0, str(REPO_ROOT))

import casadi as ca  # noqa: E402
from acados_template import AcadosOcpSolver  # noqa: E402
from codegen.models import (  # noqa: E402
    MODEL_REGISTRY,
    RNG_SEED,
    barrier_expression,
    discretise,
)
from codegen.generate_mpc_cbf_solver import build_ocp  # noqa: E402

# Fixed seed, printed in the first cell (ground rule: replayable results).
print(f"seed: {RNG_SEED:#x}   (codegen.models.RNG_SEED)")

HERE = REPO_ROOT / "analysis"
FIGDIR = HERE / "figures"
FIGDIR.mkdir(parents=True, exist_ok=True)

# --- Scenario constants (imported from codegen, not retyped) --------------
N_OBSTACLES = 8
DT = 0.1
GAMMA = 0.3                       # fixture default
GOAL_TOL = 0.05
MAX_STEPS = 120

spec2d = MODEL_REGISTRY["double_integrator_2d"]()
_xs = ca.SX.sym("x", spec2d.nx)
_os = ca.SX.sym("o", 7)
_h_fn = ca.Function("h", [_xs, _os], [barrier_expression(spec2d, _xs, _os)])


def barrier(x: np.ndarray, obs7: np.ndarray) -> float:
    """h(x) for one obstacle given its 7-slot parameter block."""
    return float(_h_fn(x, obs7))


F_di = discretise(spec2d, DT, "exact")  # exact ZOH, bit-identical to the solvers


def step_di(x: np.ndarray, u: np.ndarray) -> np.ndarray:
    return np.asarray(F_di(x, u)).flatten()


def obs7_of(obs: dict) -> np.ndarray:
    return np.array([obs["position"][0], obs["position"][1], 0.0,
                     0.0, 0.0, 0.0, obs["radius"]])


def parameter_vector(stage: int, obstacles: list, gamma: float,
                     n_obstacles: int = N_OBSTACLES) -> np.ndarray:
    """[o_0(7), ..., o_{n-1}(7), gamma]; layout per section 5.3 of the design doc."""
    p = np.zeros(7 * n_obstacles + 1)
    for j in range(n_obstacles):
        if j < len(obstacles):
            p[7 * j:7 * j + 7] = obs7_of(obstacles[j])
        else:
            p[7 * j:7 * j + 3] = 1.0e6     # far-away dummy, as in the C++ prune pad
    p[7 * n_obstacles] = gamma
    return p


def set_reference(solver, x_ref: np.ndarray) -> None:
    """Constant set-point tracked by every stage (inputs penalised at 0)."""
    N = solver.acados_ocp.dims.N
    yref = np.concatenate([x_ref, np.zeros(2)])
    for k in range(N):
        solver.set(k, "yref", yref)
    solver.set(N, "yref", x_ref)


def coast_warm_start(solver, x0: np.ndarray) -> None:
    """Constant-x0 rollout on stages 1..N, u = 0 (the C++ runtime's cold start)."""
    N = solver.acados_ocp.dims.N
    nu_dims = solver.acados_ocp.dims.nu
    nu = nu_dims[0] if isinstance(nu_dims, (list, np.ndarray)) else nu_dims
    for k in range(1, N + 1):
        solver.set(k, "x", x0)
    for k in range(N):
        solver.set(k, "u", np.zeros(nu))


# --- Scenario generator: 100 seeded scenarios spanning positions, radii,
# starts and goals. A scenario is rejected only when the start or the goal
# is inside the inflated obstacle (h < 0.01), so the hard cases — a direct
# path through the obstacle region — survive.
START_CORNERS = np.array([[0.0, 0.0], [0.2, 0.0], [0.0, 0.2], [0.2, 0.2]])
GOAL_CORNERS = np.array([[1.0, 1.0], [1.2, 1.0], [1.0, 1.2], [1.2, 1.2]])
EGO_RADIUS, SAFETY_MARGIN = 0.15, 0.05

rng = np.random.default_rng(RNG_SEED)
scenarios: list[dict] = []
while len(scenarios) < 100:
    start = START_CORNERS[rng.integers(0, len(START_CORNERS))].copy()
    goal = GOAL_CORNERS[rng.integers(0, len(GOAL_CORNERS))].copy()
    pos = rng.uniform(0.3, 1.2, 2)
    r = rng.uniform(0.12, 0.30)
    r_eff = r + EGO_RADIUS + SAFETY_MARGIN
    o7 = np.array([pos[0], pos[1], 0.0, 0.0, 0.0, 0.0, r_eff])
    if barrier(np.r_[start, 0, 0], o7) < 0.01 or barrier(np.r_[goal, 0, 0], o7) < 0.01:
        continue
    scenarios.append({
        "id": len(scenarios),
        "start": start.tolist(),
        "goal": goal.tolist(),
        "obstacle": {"position": pos.tolist(), "radius": r, "r_eff": r_eff},
    })

scenarios_json = HERE / "scenarios.json"
scenarios_json.write_text(json.dumps({
    "seed": RNG_SEED, "dt": DT, "gamma": GAMMA, "ego_radius": EGO_RADIUS,
    "safety_margin": SAFETY_MARGIN, "max_steps": MAX_STEPS,
    "scenarios": scenarios,
}, indent=1))
print(f"{len(scenarios)} scenarios -> analysis/scenarios.json")


In [ ]:
# --- Solver factory: identical plant / cost / barrier; only the horizon
# differs (N = 1 for the CBF-QP filter, N = 15 for the MPC-CBF). Generated
# code goes under results/ (gitignored).
GEN_DIR = REPO_ROOT / "results" / "acados_cbfqp_comparison"
_solver_cache: dict = {}


def make_solver(horizon: int):
    if horizon in _solver_cache:
        return _solver_cache[horizon]
    ocp = build_ocp(model_name="double_integrator_2d", horizon=horizon, dt=DT,
                    variant="fixed_decay", n_obstacles=N_OBSTACLES)
    ocp.solver_options.nlp_solver_max_iter = 100
    ocp.solver_options.print_level = 0   # silent; failure is measured, not printed
    name = f"cbfqp_comparison_di2d_N{horizon}_fixed_decay"
    ocp.name = name
    out = GEN_DIR / name
    out.mkdir(parents=True, exist_ok=True)
    ocp.code_gen_options.json_file = str(out / f"{name}.json")
    ocp.code_gen_options.code_export_directory = str(out)
    _solver_cache[horizon] = AcadosOcpSolver(ocp, generate=True, build=True)
    return _solver_cache[horizon]


solver_cbfqp = make_solver(1)     # the myopic one-step filter
solver_mpccbf = make_solver(15)   # the full lookahead
print("CBF-QP  (N=1):", solver_cbfqp.acados_ocp.model.name,
      "| nu =", solver_cbfqp.acados_ocp.dims.nu)
print("MPC-CBF (N=15):", solver_mpccbf.acados_ocp.model.name,
      "| nu =", solver_mpccbf.acados_ocp.dims.nu)

# Identical input space, different horizon: the ONLY structural difference.
assert solver_cbfqp.acados_ocp.dims.N == 1 and solver_mpccbf.acados_ocp.dims.N == 15
assert solver_cbfqp.acados_ocp.dims.nu == solver_mpccbf.acados_ocp.dims.nu
print("assert OK: N = 1 vs N = 15, identical nu")


In [ ]:
# --- Closed loop with brake fallback --------------------------------------
# On an infeasible solve the applied input is u = 0 (the node's brake fallback)
# and the loop keeps stepping, so the *cost* of the failure — coasting into the
# obstacle, h < 0 — is measured rather than hidden by stopping.

# acados prints "QP solver returned error status" to stdout on every failed
# solve; failures are counted in `statuses`, so we silence fd 1 and fd 2 for
# the duration of each solve (a failing run prints hundreds of lines otherwise).
class _silence_io:
    def __enter__(self):
        self._saved = (os.dup(1), os.dup(2))
        self._null = os.open(os.devnull, os.O_WRONLY)
        os.dup2(self._null, 1)
        os.dup2(self._null, 2)
        return self

    def __exit__(self, *exc):
        os.dup2(self._saved[0], 1)
        os.dup2(self._saved[1], 2)
        os.close(self._saved[0])
        os.close(self._saved[1])
        os.close(self._null)


def run_controller(solver, scen: dict, max_steps: int = MAX_STEPS,
                   gamma: float = GAMMA):
    N = solver.acados_ocp.dims.N
    obs = {"position": np.array(scen["obstacle"]["position"] + [0.0]),
           "velocity": np.zeros(3), "radius": scen["obstacle"]["r_eff"],
           "is_dynamic": False}
    x0 = np.array(scen["start"] + [0.0, 0.0])
    goal = np.array(scen["goal"] + [0.0, 0.0])
    X, U, H, statuses, t_all, t_ok = [], [], [], [], [], []
    x = x0.copy()
    reached = False
    for _ in range(max_steps):
        for j in range(N + 1):
            solver.set(j, "p", parameter_vector(j, [obs], gamma))
        set_reference(solver, goal)
        solver.set(0, "lbx", x)
        solver.set(0, "ubx", x)
        solver.set(0, "x", x)
        coast_warm_start(solver, x)
        t0 = time.perf_counter()
        with _silence_io():
            st = solver.solve()
        dt_ms = (time.perf_counter() - t0) * 1e3
        statuses.append(st)
        t_all.append(dt_ms)
        u = np.zeros(2)
        if st == 0:
            u = np.array(solver.get(0, "u")).flatten()[:2]
            t_ok.append(dt_ms)
        X.append(x.copy())
        H.append(barrier(x, obs7_of(obs)))
        U.append(u)
        x = step_di(x, u)
        if np.linalg.norm(x[:2] - goal[:2]) <= GOAL_TOL:
            reached = True
            break
    return dict(X=np.array(X), U=np.array(U), H=np.array(H),
                statuses=np.array(statuses), times_all=np.array(t_all),
                times_ok=np.array(t_ok), reached=reached)


def metrics(run: dict, max_steps: int = MAX_STEPS) -> dict:
    """Per-run metrics: collision (min h < 0), infeasible steps, time-to-goal,
    path length, control effort, mean and p95 solve time. Solve time is
    reported twice: per *successful* control decision (headline — this is what
    a controller comparison means) and over all solves (failed SQP iterations
    are part of the run's real cost)."""
    return dict(
        reached=bool(run["reached"]),
        steps=int(len(run["X"])),
        min_h=float(run["H"].min()),
        collision=bool(run["H"].min() < 0.0),
        infeasible=int((run["statuses"] != 0).sum()),
        path_length=float(np.linalg.norm(np.diff(run["X"][:, :2], axis=0), axis=1).sum()),
        control_effort=float((run["U"] ** 2).sum()),
        mean_ms_ok=float(run["times_ok"].mean()) if len(run["times_ok"]) else float("nan"),
        p95_ms_ok=float(np.percentile(run["times_ok"], 95)) if len(run["times_ok"]) else float("nan"),
        mean_ms_all=float(run["times_all"].mean()),
        p95_ms_all=float(np.percentile(run["times_all"], 95)),
    )


# --- Run both controllers over all 100 scenarios ---------------------------
rows = []
for scen in scenarios:
    m1 = metrics(run_controller(solver_cbfqp, scen))
    m15 = metrics(run_controller(solver_mpccbf, scen))
    rows.append({"scenario": scen["id"],
                 **{f"cbfqp_{k}": v for k, v in m1.items()},
                 **{f"mpccbf_{k}": v for k, v in m15.items()}})
df = pd.DataFrame(rows)
df.to_csv(HERE / "results_comparison.csv", index=False)
print(f"{len(df)} scenarios x 2 controllers -> analysis/results_comparison.csv")


## Results

Measured over the 100 seeded scenarios (identical plant, cost, barrier; $N=1$ vs $N=15$):

| metric | CBF-QP ($N=1$) | MPC-CBF ($N=15$) |
|---|---|---|
| runs violating safety ($\min_k h < 0$) | 49 / 100 | 0 / 100 |
| runs reaching the goal | 99 / 100 | 92 / 100 |
| median steps to goal (mutual completions) | 66 | 26 |
| solve-time ratio, mean / p95 | 1× | 14–20× / 18–25× (run-dependent) |

The solve-time numbers vary between runs (they are wall-clock times, and the machine is shared),
so the exact values are printed by the assert cell below and the invariant — the ratio lies in the
10–50× band the design doc predicts — is enforced there. Everything else is seed-deterministic and
re-runs byte-identically.

1. **Collision counts.** CBF-QP violates $h<0$ in 49/100 runs; MPC-CBF in 0/100. The mechanism
   is the horizon: the one-step filter cannot see the obstacle closing in. When the one-step DCBF
   finally becomes infeasible the ego is still moving; the brake fallback lets it coast into the
   obstacle. MPC-CBF sees the obstacle 15 steps ahead, slows early and never enters it — and when
   it does fail, it fails stopped, short of the obstacle, safely.
2. **Solve time.** The myopic filter wins, plainly: measured ratios land in the 10–50× band
   (mean ~15–20×, p95 ~18–25× in the runs we measured; the assert below pins the current run).
   Including failed solves the ratio is ~60×: a failing $N=15$ solve spends its whole SQP budget,
   so failures are expensive on top of being failures.
3. **Where MPC-CBF loses.** In 8 scenarios MPC-CBF never reaches the goal while the myopic
   filter does. The mechanism is the recursive-feasibility shrink with the horizon: the $N=15$
   constraint set (DCBF rows at every prediction stage) becomes infeasible from a state the $N=1$
   problem still solves, and the longer-horizon planner — which committed early to one side of
   the obstacle — is stranded. The horizon is not uniformly a gift.
4. **The honest balance.** MPC-CBF is the safer controller (0 vs 49 violations) and the more
   efficient one when it completes (26 vs 66 median steps) — but it is ~15–20× slower and loses
   on completion in 8 scenarios. A one-sided plot would have said "MPC-CBF is better"; this one
   says where each fails, and why.


In [ ]:
# --- The claims, each an assert -------------------------------------------
ok = df.dropna(subset=["cbfqp_mean_ms_ok", "mpccbf_mean_ms_ok"])
ratio_mean = float(ok["mpccbf_mean_ms_ok"].mean() / ok["cbfqp_mean_ms_ok"].mean())
ratio_p95 = float(ok["mpccbf_p95_ms_ok"].mean() / ok["cbfqp_p95_ms_ok"].mean())
print(f"successful-step solve time: CBF-QP {ok['cbfqp_mean_ms_ok'].mean():.3f} ms | "
      f"MPC-CBF {ok['mpccbf_mean_ms_ok'].mean():.3f} ms | ratio {ratio_mean:.1f}x")
print(f"successful-step p95:        CBF-QP {ok['cbfqp_p95_ms_ok'].mean():.3f} ms | "
      f"MPC-CBF {ok['mpccbf_p95_ms_ok'].mean():.3f} ms | ratio {ratio_p95:.1f}x")
ratio_all = float(df["mpccbf_mean_ms_all"].mean() / df["cbfqp_mean_ms_all"].mean())
print(f"all-solves (incl. failures): ratio {ratio_all:.1f}x")
assert 10.0 <= ratio_mean <= 50.0, f"solve-time ratio {ratio_mean:.1f}x outside 10-50x band"
assert ratio_p95 >= 10.0, f"p95 ratio {ratio_p95:.1f}x"
print("assert OK: myopic filter wins on solve time by 10-50x (measured "
      f"{ratio_mean:.1f}x mean, {ratio_p95:.1f}x p95)")

# MPC-CBF loses somewhere: it fails to reach while the myopic filter reaches.
lost = df[(~df["mpccbf_reached"]) & (df["cbfqp_reached"])]
print(f"MPC-CBF loss scenarios (mpccbf stuck, cbfqp reached): {len(lost)} "
      f"{lost['scenario'].tolist()}")
assert len(lost) >= 1, "no scenario where MPC-CBF loses - widen the distribution"
print("assert OK: >=1 scenario where MPC-CBF loses")

# Two-sided honesty: the myopic filter also fails, and violates more often.
n_viol_cbfqp = int(df["cbfqp_collision"].sum())
n_viol_mpccbf = int(df["mpccbf_collision"].sum())
print(f"collisions (min h < 0): CBF-QP {n_viol_cbfqp} | MPC-CBF {n_viol_mpccbf}")
assert n_viol_cbfqp > 0 and n_viol_mpccbf < n_viol_cbfqp
print("assert OK: CBF-QP violates safety in some scenarios; MPC-CBF in fewer")

# Path efficiency among mutual completions.
both = df[df["cbfqp_reached"] & df["mpccbf_reached"]]
med1, med15 = float(both["cbfqp_steps"].median()), float(both["mpccbf_steps"].median())
print(f"mutual completions {len(both)}: median steps CBF-QP {med1:.0f} vs MPC-CBF {med15:.0f}")
assert med15 < med1, (med1, med15)
print("assert OK: MPC-CBF is more step-efficient when both complete")

# --- Summary table + box plots --------------------------------------------
summary = pd.DataFrame({
    "controller": ["CBF-QP (N=1)", "MPC-CBF (N=15)"],
    "safety violations": [n_viol_cbfqp, n_viol_mpccbf],
    "reached goal": [int(df["cbfqp_reached"].sum()), int(df["mpccbf_reached"].sum())],
    "median steps (mutual)": [med1, med15],
    "mean solve ms (ok)": [ok["cbfqp_mean_ms_ok"].mean(), ok["mpccbf_mean_ms_ok"].mean()],
    "p95 solve ms (ok)": [ok["cbfqp_p95_ms_ok"].mean(), ok["mpccbf_p95_ms_ok"].mean()],
})
print(summary.to_string(index=False))

fig, axes = plt.subplots(2, 2, figsize=(9, 7))
ax = axes[0, 0]
ax.boxplot([df["cbfqp_mean_ms_ok"].dropna(), df["mpccbf_mean_ms_ok"].dropna()],
           tick_labels=["CBF-QP", "MPC-CBF"])
ax.set_yscale("log")
ax.set_title("solve time per successful step (log ms)")
ax = axes[0, 1]
ax.boxplot([df[df["cbfqp_reached"]]["cbfqp_steps"],
            df[df["mpccbf_reached"]]["mpccbf_steps"]],
           tick_labels=["CBF-QP", "MPC-CBF"])
ax.set_title("steps to goal (reached runs)")
ax = axes[1, 0]
ax.boxplot([df["cbfqp_min_h"], df["mpccbf_min_h"]], tick_labels=["CBF-QP", "MPC-CBF"])
ax.axhline(0.0, color="k", lw=0.8)
ax.set_title("min h over run (collision below 0)")
ax = axes[1, 1]
ax.scatter(df["cbfqp_path_length"], df["cbfqp_control_effort"],
           s=8, alpha=0.5, label="CBF-QP")
ax.scatter(df["mpccbf_path_length"], df["mpccbf_control_effort"],
           s=8, alpha=0.5, label="MPC-CBF")
ax.set_xlabel("path length (m)")
ax.set_ylabel("control effort")
ax.set_title("path length vs control effort")
ax.legend()
fig.tight_layout()
fig.savefig(FIGDIR / "comparison_summary.png", dpi=130)
plt.close(fig)
print("saved figures/comparison_summary.png")


In [ ]:
# --- Side-by-side animation ----------------------------------------------
# Same seed, same scenario, one controller colliding: pick the first scenario
# where CBF-QP violates h < 0 while MPC-CBF reaches the goal safely.
from matplotlib.animation import FuncAnimation, PillowWriter  # noqa: E402

gif_rows = df[(df["cbfqp_collision"]) & (df["mpccbf_reached"]) & (df["mpccbf_min_h"] >= 0.0)]
assert len(gif_rows) >= 1, "no scenario where CBF-QP collides and MPC-CBF holds"
gif_row = gif_rows.iloc[0]
scen = scenarios[int(gif_row["scenario"])]
print(f"GIF scenario {scen['id']}: start {scen['start']} goal {scen['goal']} "
      f"obstacle {np.round(scen['obstacle']['position'], 3)} r_eff {scen['obstacle']['r_eff']:.3f}")

r_cbfqp = run_controller(solver_cbfqp, scen)
r_mpccbf = run_controller(solver_mpccbf, scen)
minh1, minh15 = float(r_cbfqp["H"].min()), float(r_mpccbf["H"].min())
print(f"CBF-QP:  {len(r_cbfqp['X'])} steps, min h {minh1:+.4f}")
print(f"MPC-CBF: {len(r_mpccbf['X'])} steps, min h {minh15:+.4f}")
assert minh1 < 0.0 and minh15 >= 0.0, "GIF must show one controller colliding"

# Cap at ~6 s at 15 fps (90 frames); the longer trajectory is subsampled.
nframes = min(90, max(len(r_cbfqp["X"]), len(r_mpccbf["X"])))


def frame_idx(L: int) -> np.ndarray:
    return np.round(np.linspace(0, L - 1, nframes)).astype(int)


i1, i15 = frame_idx(len(r_cbfqp["X"])), frame_idx(len(r_mpccbf["X"]))
obs_pos = np.array(scen["obstacle"]["position"])
r_eff = scen["obstacle"]["r_eff"]
fig, (ax1, ax15) = plt.subplots(1, 2, figsize=(8.0, 4.0))


def draw(ax, X, H, i, title, color):
    ax.clear()
    ax.add_patch(plt.Circle(tuple(obs_pos), r_eff, color="0.35", alpha=0.20))
    ax.plot(X[:i + 1, 0], X[:i + 1, 1], "-", color=color, lw=1.2)
    ax.plot(X[i, 0], X[i, 1], "o", color=color, ms=6)
    ax.plot(scen["goal"][0], scen["goal"][1], "*", color="tab:green", ms=11)
    ax.plot(scen["start"][0], scen["start"][1], "s", color="0.3", ms=5)
    ax.set_xlim(-0.15, 1.45)
    ax.set_ylim(-0.15, 1.45)
    ax.set_aspect("equal")
    ax.set_title(f"{title}   h = {H[i]:+.4f}", fontsize=10)


def update(f: int):
    draw(ax1, r_cbfqp["X"], r_cbfqp["H"], i1[f], "CBF-QP (N=1)", "tab:blue")
    draw(ax15, r_mpccbf["X"], r_mpccbf["H"], i15[f], "MPC-CBF (N=15)", "tab:red")
    fig.suptitle(f"scenario {scen['id']}: same seed, same obstacle, one controller colliding",
                 fontsize=10)


anim = FuncAnimation(fig, update, frames=nframes, interval=1000 // 15)
gif_path = REPO_ROOT / "media" / "cbfqp_vs_mpccbf_side_by_side.gif"
anim.save(gif_path, writer=PillowWriter(fps=15))
plt.close(fig)
size_mb = gif_path.stat().st_size / 1e6
print(f"saved {gif_path} ({size_mb:.2f} MB, {nframes} frames @ 15 fps = {nframes / 15:.1f} s)")
assert size_mb <= 8.0, f"GIF {size_mb:.2f} MB exceeds the 8 MB cap"
assert nframes / 15 <= 8.0
print("assert OK: GIF <= 8 MB, <= 8 s, >= 15 fps, every frame carries h(x)")
